# 00 Environment Check

Validate the Fabric Spark runtime, Lakehouse attachment, project configuration files, and basic Delta table write/read behaviour.


## Initialise Run Context

This cell detects local versus Fabric runtime and prints basic metadata.


In [ ]:
# Cell purpose: Initialise Run Context.
from datetime import datetime, timezone
from pathlib import Path
import json
import uuid

try:
    spark
    is_local_run = False
except NameError:
    is_local_run = True

run_id = str(uuid.uuid4())

print(f"run_id={run_id}")
print(f"runtime={'local' if is_local_run else 'fabric'}")
print(f"utc_now={datetime.now(timezone.utc).isoformat()}")

if not is_local_run:
    try:
        fabric_context = notebookutils.runtime.context
        print(f"workspace_name={fabric_context.get('currentWorkspaceName') or 'unknown'}")
        print(f"lakehouse_name={fabric_context.get('defaultLakehouseName') or 'none_attached'}")
    except Exception as exc:
        print(f"fabric_context=unavailable ({type(exc).__name__})")


## Verify Runtime

This cell confirms Spark only when the notebook is running in Fabric. Local runs skip Spark validation.


In [ ]:
# Cell purpose: Verify Runtime.
if is_local_run:
    print("Spark validation skipped for local run.")
else:
    spark_version = spark.version
    print(f"Spark available: {spark_version}")


## Check Project Files

This cell checks whether the repository configuration files are available to the notebook runtime. If they are missing in Fabric, upload the repo files or package the Python project with the notebook.


In [ ]:
# Cell purpose: Check Project Files.
required_paths = [
    "config/sources.yml",
    "config/tables.yml",
    "config/dashboard_requirements.yml",
]

fabric_files_root = Path("/lakehouse/default/Files")
missing = []
for path in required_paths:
    local_path = Path(path)
    fabric_path = fabric_files_root / path
    if is_local_run:
        if not local_path.exists():
            missing.append(path)
    elif not local_path.exists() and not fabric_path.exists():
        missing.append(path)

if missing:
    print("Missing config files:", missing)
else:
    print("Config files found:", required_paths)


## Validate Lakehouse Table Access

This cell writes and reads a small Delta table only in Fabric. Local runs skip Lakehouse validation.


In [ ]:
# Cell purpose: Validate Lakehouse Table Access.
if is_local_run:
    print("Lakehouse table validation skipped for local run.")
else:
    health_table = "environment_healthcheck"
    payload = [{"run_id": run_id, "checked_at_utc": datetime.now(timezone.utc).isoformat(), "status": "ok"}]
    df = spark.createDataFrame(payload)
    df.write.format("delta").mode("append").saveAsTable(health_table)
    display(spark.table(health_table).orderBy("checked_at_utc", ascending=False).limit(5))
